# Colab: pretrained-model features (ESM-2 650M and ESM-IF1)

This notebook computes the two GPU-heavy feature sets of the project **once**, and saves them as files that the rest of the
pipeline (CPU, local) reads. No training happens here.

| Output file | Model | What it contains | Used by |
|---|---|---|---|
| `esm2_650M.parquet` | ESM-2 650M (sequence) | per mutation: zero-shot log-likelihood ratio, wild-type site embedding, mutant − wild-type embedding (1,280 dims each) | model A with the larger language model |
| `esmif1.parquet` | ESM-IF1 (inverse folding, structure) | per mutation: site and whole-chain log-likelihood ratios **in the complex**, **unbound** and their difference (**interface**), plus the site's structure embedding in both contexts (512 dims each) | models D and E |

**Why these two models** (see README §4.1): ESM-2 is the sequence modality. ESM-IF1 reads the experimental backbone of the complex and answers
"does this residue fit this 3D environment?". Scoring the mutated chain with and without its binding partner isolates what the partner contributes.

**Before you start**
1. Locally, run `python scripts/make_colab_bundle.py`. It writes `colab_bundle.zip` (~7 MB: code, cleaned data table, the 55 antibody–antigen structures).
2. Upload it to Google Drive as `MyDrive/converge_ha/colab_bundle.zip` (or change `BUNDLE` below).
3. Runtime → Change runtime type → **T4 GPU** (any GPU works).

**Runtime on a T4:** about 5 min of installs and downloads, ESM-2 650M about 3–5 min, ESM-IF1 about 5–10 min.

## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime -> Change runtime type -> T4 GPU. (It still runs on CPU, but ESM-IF1 takes about 2.5 hours.)")

## 2. Get the project files from Google Drive

In [ ]:
import os, zipfile
from google.colab import drive

drive.mount("/content/drive")
BUNDLE = "/content/drive/MyDrive/converge_ha/colab_bundle.zip"   # change if you put it elsewhere
OUT_DIR = "/content/drive/MyDrive/converge_ha"                    # results are copied back here
WORK = "/content/converge_ha"

with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)), "|", len(os.listdir("data/SKEMPI2_PDBs/PDBs")), "structure files")

## 3. Install the dependencies

- `fair-esm` provides ESM-IF1. `transformers` (pre-installed on Colab) provides ESM-2.
- **biotite ≥ 1.0** is required on current Colab (NumPy 2). It renamed a function fair-esm imports. `src/features/inverse_folding.py`
  restores the old name, so always import that module **before** `esm.inverse_folding`.
- **No `torch_scatter` needed.** It is a compiled package that often has no wheel for Colab's newest torch. fair-esm only uses one of its
  functions, so the project module installs an equivalent pure-PyTorch version when the package is missing (tested to give identical
  results). `torch_geometric` is pure Python.

In [ ]:
!pip install -q fair-esm "biotite>=1.0" torch_geometric biopython

In [ ]:
import importlib, sys
sys.path.insert(0, WORK)
import src.features.inverse_folding as inf      # applies the biotite and torch_scatter compatibility fixes
import esm.inverse_folding, biotite, numpy, transformers
print("biotite", biotite.__version__, "| numpy", numpy.__version__, "| transformers", transformers.__version__)
print("torch_scatter:", "real package" if hasattr(sys.modules["torch_scatter"], "__file__") else "pure-PyTorch fallback")

## 4. ESM-2 650M sequence features

Same script as the local 35M run; only the model name changes. For each mutation, only the mutated chain goes into ESM-2 (the model was trained on single chains).

In [ ]:
!python scripts/04_esm2.py facebook/esm2_t33_650M_UR50D

## 5. ESM-IF1 structure features

For each mutation, the wild-type backbone is scored in two contexts:
- **complex:** the mutated chain plus all chains of both partners
- **unbound:** the mutated chain plus only its own partner's other chains (e.g. the light chain for a heavy-chain mutation)

ESM-IF1 decodes left to right, so two scores are kept:
- the **site** log-likelihood ratio, log p(mut) − log p(wt) at the mutated position
- the **whole-chain** ratio, the standard ESM-IF1 mutation score, which also counts the residues decoded after the mutation

`interface = complex − unbound` is the part attributable to the binding partner.

In [ ]:
# checkpoints every 25 mutations: if the session disconnects, rerun this cell (after steps 2-3) to resume
!python scripts/08_esmif1.py

## 6. Sanity checks

What to expect:
- both files have 668 rows and no missing values
- the ESM-IF1 **interface** scores correlate **negatively** with ΔΔG (a mutant that fits the interface worse binds worse). A 40-mutation test
  on CPU gave about −0.34.
- ESM-2 35M's zero-shot score had no signal (≈ 0). We want to see whether 650M changes that.

In [ ]:
import pandas as pd
KEY = ["complex", "Mutation(s)_cleaned"]
dd = pd.read_parquet("data/processed/skempi_abag_dedup.parquet")
v1 = dd[~dd["censored"] & (dd["n_mut"] == 1)][KEY + ["ddG"]]

esm2 = pd.read_parquet("data/processed/esm2_650M.parquet")
esmif = pd.read_parquet("data/processed/esmif1.parquet")
for name, df in [("esm2_650M", esm2), ("esmif1", esmif)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns, missing values: {int(df.isna().sum().sum())}")

m = v1.merge(esm2[KEY + ["llr"]], on=KEY).merge(esmif[KEY + [c for c in esmif.columns if c.startswith(("if_site", "if_seq"))]], on=KEY)
print("\nSpearman with ddG (all 668 mutations):")
print(m.drop(columns=KEY).corr("spearman")["ddG"].drop("ddG").round(3).to_string())

## 7. Save the results to Google Drive

Then, locally: copy both files into `data/processed/` of the project. `esm2_650M.parquet` is about 15 MB and `esmif1.parquet` about 3 MB.

In [ ]:
import shutil
for f in ["esm2_650M.parquet", "esmif1.parquet"]:
    shutil.copy(f"data/processed/{f}", f"{OUT_DIR}/{f}")
    print("saved", f"{OUT_DIR}/{f}")
# Or download directly:
# from google.colab import files; files.download("data/processed/esmif1.parquet"); files.download("data/processed/esm2_650M.parquet")